In [1]:
import sys
import os

# Add the project root directory to Python's path
# This allows us to import 'qgcn_lib' from inside 'examples/notebooks'
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)



In [2]:
import torch
import os
import math
import tqdm
from torch_geometric.nn import DeepGraphInfomax

# --- 1. Import from our Library ---
from qgcn_lib.datasets import ExperimentDataset 
from qgcn_lib.nn import HybridQGCNConv, SummaryMLP
from qgcn_lib.utils import set_all_seeds, feature_shuffling_corruption

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
dataset_name = 'cora' 

# Construct the path relative to this script
# We assume data is in a './data' folder next to this script
current_dir = os.getcwd()
file_path = os.path.join(current_dir, '..', 'data', f'{dataset_name}.pt')

print(f"--> Initializing Library Wrapper for: {dataset_name}")

# USE THE WRAPPER: This creates a PyG-compatible dataset object
try:
    dataset = ExperimentDataset(root=os.path.join(current_dir, 'data'), file_path=file_path)
    data = dataset[0]  # Get the graph object
    
    # Move to device immediately
    data.x = data.x.to(device)
    data.edge_index = data.edge_index.to(device)
    
    print(f"--> Data Loaded Successfully via qgcn_lib")
    print(f"    Nodes: {data.num_nodes} | Features: {data.num_features}")
    
except FileNotFoundError:
    print(f"\n[ERROR] Could not find {dataset_name}.pt")
    print(f"Please ensure you have placed the file at: {file_path}")
    exit()


--> Initializing Library Wrapper for: cora
--> Data Loaded Successfully via qgcn_lib
    Nodes: 2708 | Features: 1433


In [5]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [6]:
in_channels = data.x.size(1)
hidden_channels = math.ceil(math.log2(in_channels))

print(f"--> Feature Dimension (d): {in_channels}")
print(f"--> Quantum Latent Space: {hidden_channels} Qubits")

def train_quantum_dgi(model, features, edge_index, epochs):
    """
    Training loop with Group-Wise Learning Rates.
    """
    model.to(device)
    features = features.to(device)
    edge_index = edge_index.to(device)

    param_groups = []
    
   
    if hasattr(model.encoder, 'qc'):
        param_groups.append({'params': model.encoder.qc.parameters(), 'lr': 0.01})
    if hasattr(model.encoder, 'local_mp'):
        param_groups.append({'params': model.encoder.local_mp.parameters(), 'lr': 0.01})

    classical_params = []
    if hasattr(model.encoder, 'q_proj'): classical_params.extend(model.encoder.q_proj.parameters())
    if hasattr(model.encoder, 'bias'): classical_params.append(model.encoder.bias)
    if hasattr(model.encoder, 'prelu'): classical_params.extend(model.encoder.prelu.parameters())
    if hasattr(model, 'summary'): classical_params.extend(model.summary.parameters())
    
    param_groups.append({'params': classical_params, 'lr': 0.001})

    # Initialize Optimizer with the grouped parameters
    optimizer = torch.optim.Adam(param_groups)

    # Standard DGI Training Loop
    print(f"--> Starting Training for {epochs} epochs...")
    for epoch in tqdm.tqdm(range(epochs), desc="Training"):
        model.train()
        optimizer.zero_grad()
        
        # Forward pass returns: Real Embeddings, Corrupted Embeddings, Global Summary
        pos_z, neg_z, summary = model(features, edge_index)
        
        loss = model.loss(pos_z, neg_z, summary)
        loss.backward()
        optimizer.step()
        
    return model

def run_experiment(features, idx_edge):
    # 1. Reproducibility
    # Essential for quantum simulations where measurement outcomes can be stochastic.
    set_all_seeds(123)
    
    num_nodes = features.size(0)
    
    # 2. Initialize the Quantum Encoder
    # - hidden_channels: Calculated dynamically (log2 d)
    # - q_depth=1: Depth of the Variational Quantum Circuit (VQC) layers
    encoder = HybridQGCNConv(
        in_channels=features.size(1), 
        points=num_nodes, 
        hidden_channels=hidden_channels, 
        q_depth=1
    )
    
    # 3. Initialize the Global Summary Network
    summary = SummaryMLP(hidden_channels)
    
    # 4. Construct the DGI Model
    # We use 'feature_shuffling_corruption' to generate negative samples.
    # The model learns by maximizing Mutual Information between:
    #   - Positive: Real local patches + Global summary
    #   - Negative: Shuffled features + Global summary
    model = DeepGraphInfomax(
        hidden_channels=hidden_channels,
        encoder=encoder,
        summary=summary,
        corruption=feature_shuffling_corruption
    )
    
    # 5. Execute Training
    # Uses the differential learning rate strategy defined earlier.
    model = train_quantum_dgi(model, features, idx_edge, epochs=50)
    
    # 6. Extract Embeddings (Z)
    model.eval()
    with torch.no_grad():
        z, _, _ = model(features, idx_edge)
        
    return z


--> Feature Dimension (d): 1433
--> Quantum Latent Space: 11 Qubits


In [7]:
z = run_experiment(data.x, data.edge_index)

--> Starting Training for 50 epochs...


Training: 100%|██████████| 50/50 [14:23<00:00, 17.28s/it]


In [8]:
from qgcn_lib.utils import perform_kmeans_clustering, visualize_embedding
from sklearn.metrics import normalized_mutual_info_score


# 1. Cluster Analysis (K-Means)
# We ask K-Means to find 3 clusters in the 4-dimensional quantum latent space
labels, z_np, score = perform_kmeans_clustering(z, 7)

# 2. Quantitative Metric (NMI)
# We compare the learned clusters against the true graph communities
# NMI = 1.0 means perfect correlation; NMI = 0.0 means random.
nmi = normalized_mutual_info_score(data.y.cpu().numpy(), labels)

print(f"--> Silhouette Score: {score:.4f}")
print(f"--> NMI Score (Cluster Quality): {nmi:.4f}")

# 3. Visualization (t-SNE)
# This generates a 2D projection of the quantum embeddings
# visualize_embedding(z_np, labels, score, 3)

Silhouette Score (k=7): 0.8941
--> Silhouette Score: 0.8941
--> NMI Score (Cluster Quality): 0.0069


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, normalized_mutual_info_score

X_train, X_test, y_train, y_test = train_test_split(
            z_np, data.y.cpu().numpy(), test_size=0.2, random_state=123, stratify=data.y.cpu().numpy())
clf = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=123)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n" + "="*40)
print(f"RESULTS FOR {dataset_name}")
print("="*40)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("="*40)


RESULTS FOR cora
Confusion Matrix:
 [[ 0 21 12 24  0 13  0]
 [ 0 14  9 12  0  8  0]
 [ 0 28 18 28  0 10  0]
 [ 0 36 24 71  0 33  0]
 [ 0 24 10 32  0 19  0]
 [ 0 20  7 12  0 21  0]
 [ 0 10  4 13  0  9  0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        70
           1       0.09      0.33      0.14        43
           2       0.21      0.21      0.21        84
           3       0.37      0.43      0.40       164
           4       0.00      0.00      0.00        85
           5       0.19      0.35      0.24        60
           6       0.00      0.00      0.00        36

    accuracy                           0.23       542
   macro avg       0.12      0.19      0.14       542
weighted avg       0.17      0.23      0.19       542



/opt/miniconda3/envs/main/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/envs/main/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/miniconda3/envs/main/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
